In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ising.model import UpdateMethod

from climate_attitudes.dataset import Dataset
from climate_attitudes.settings import Config
from climate_attitudes.visualisation import configure_mpl
from ising import Ising

configure_mpl(Path("../fonts/"))

np.set_printoptions(linewidth=200)

RANDOM_SEED = 202606170922

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(
    config,
    name="reduced_no_imputation",
    with_imputation=False,
    verbose=False,
)
_, Y, _ = dataset.indices_to_numpy(kind="time-series", binarise=True, seed=RANDOM_SEED)

replicates = 10
Y_replicated = np.empty(
    (Y.shape[0] * replicates, Y.shape[1], Y.shape[2]), dtype=np.int64
)
for r in range(replicates):
    _, Yr, _ = dataset.indices_to_numpy(
        kind="time-series", binarise=True, seed=RANDOM_SEED + r
    )
    Y_replicated[Y.shape[0] * r : Y.shape[0] * (r + 1)] = Yr

## Fit a single model with $λ\in\{10^{-6}, 10^{-1}\}$, compare sparsity

In [ ]:
np.log10(0.5)

In [ ]:
λ = 0.5
model = Ising.fit(
    Y,
    update_method=UpdateMethod.SYNCHRONOUS,
    w=λ,
    rng=RANDOM_SEED,
    self_loops=True,
    node_labels=dataset.schema.get_short_names("measurement"),
)
k = (abs(model.param_vector()) > 1e-2).sum()
print(f"Non-zero params: {k}")
model.draw();

In [ ]:
λ = 1e-6
model = Ising.fit(
    Y_replicated,
    update_method=UpdateMethod.SYNCHRONOUS,
    w=λ,
    rng=RANDOM_SEED,
    self_loops=True,
    node_labels=dataset.schema.get_short_names("measurement"),
)
k = (abs(model.param_vector()) > 1e-2).sum()
print(f"Non-zero params: {k}")
model.draw();

In [ ]:
λ = 1e-2
model = Ising.fit(
    Y,
    update_method=UpdateMethod.SYNCHRONOUS,
    w=λ,
    rng=RANDOM_SEED,
    self_loops=True,
    node_labels=dataset.schema.get_short_names("measurement"),
)
k = (abs(model.param_vector()) > 1e-2).sum()
print(f"Non-zero params: {k}")
model.draw();

In [ ]:
λ = 1e-2
model = Ising.fit(
    Y_replicated,
    update_method=UpdateMethod.SYNCHRONOUS,
    w=λ,
    rng=RANDOM_SEED,
    self_loops=True,
    node_labels=dataset.schema.get_short_names("measurement"),
)
k = (abs(model.param_vector()) > 1e-2).sum()
print(f"Non-zero params: {k}")
model.draw();

In [ ]:
λ = 1e-2
model2 = Ising.fit(
    Y,
    update_method=UpdateMethod.SYNCHRONOUS,
    w=λ,
    rng=RANDOM_SEED,
    self_loops=True,
    node_labels=dataset.schema.get_short_names("measurement"),
)
k = (abs(model2.param_vector()) > 0.01).sum()
print(f"Non-zero params: {k}")
model2.draw(use_layout_from=model, vlim_j=(-0.5, 0.5))

In [ ]:
fig, ax = plt.subplots()
sort_idx = np.argsort(model2.j.flatten())
ax.scatter(np.arange(64), model2.j.flatten()[sort_idx])

In [ ]:
model2.j

In [ ]:
model2.j

In [ ]:
model2.h

In [ ]:
model.node_labels

In [ ]:
model.j[1, 6]

## Cross-validation: Average validation likelihood across splits

Pre-calculate splits

In [ ]:
rng = np.random.default_rng(RANDOM_SEED + 1)
n_splits = 10
val_size = Y.shape[0] // n_splits

idxes_shuffled = rng.choice(np.arange(Y.shape[0]), size=Y.shape[0], replace=False)
splits = []
for i in range(n_splits):
    idxes_fit = np.concatenate(
        (idxes_shuffled[: i * val_size], idxes_shuffled[(i + 1) * val_size :])
    )
    idxes_val = idxes_shuffled[i * val_size : (i + 1) * val_size]
    splits.append((idxes_fit, idxes_val))

For each $\lambda$, fit model on each split, calculate validation likelihood, average across splits.

In [ ]:
n_lambdas = 21
λ = np.logspace(-3, 0, n_lambdas, base=10)
likelihood = np.empty_like(λ)

X = np.ones((Y.shape[0] - val_size, 2))

for i in range(n_lambdas):
    split_likelihoods = np.empty(n_splits, dtype=np.float64)
    for s_i, (idxes_fit, idxes_val) in enumerate(splits):
        model = Ising.fit(
            Y[idxes_fit],
            update_method=UpdateMethod.SYNCHRONOUS,
            w=λ[i],
            rng=RANDOM_SEED * i + s_i,
        )
        val_nll = model.time_series_nll_sync(
            Y[idxes_val], X, model.h, model.j, model.adj
        )
        split_likelihoods[s_i] = np.exp(-1 * val_nll)
    likelihood[i] = split_likelihoods.mean()

In [ ]:
fig, ax = plt.subplots(figsize=(3, 2), constrained_layout=True)
ax.plot(λ, likelihood, linewidth=1)
ax.scatter(λ, likelihood, s=10, linewidths=0.7)
ax.set_xscale("log")

## Information criterion: Choose $\lambda$ such that average BIC is minimised.

Only consider non-zero parameters in BIC calculation.

In [ ]:
72 * np.log(Y.shape[0])

In [ ]:
model.h

In [ ]:
model.j

In [ ]:
λ

In [ ]:
repeats = 10
n_lambdas = 10
λ = np.logspace(-4, -1.8, n_lambdas, base=10)

X = np.ones((Y.shape[0], 2))

bic = np.empty((n_lambdas, repeats), dtype=np.float64)
for i in range(n_lambdas):
    nonzero_i = []
    for repeat in range(repeats):
        _, Yi, _ = dataset.indices_to_numpy(
            "time-series", binarise=True, seed=RANDOM_SEED + repeat
        )
        model = Ising.fit(
            Yi,
            update_method=UpdateMethod.SYNCHRONOUS,
            w=λ[i],
            rng=RANDOM_SEED + repeat,
            self_loops=True,
        )
        # model.h[abs(model.h) < 0.01] = 0
        # model.adj[abs(model.j) < 0.01] = 0
        # model.j[abs(model.j) < 0.01] = 0
        nll = model.time_series_nll_sync(Yi, X, model.h, model.j, model.adj)
        # print(nll)
        log_likelihood = -1 * Yi.shape[0] * Yi.shape[1] * nll
        non_zero_params = (abs(model.h) > 1e-2).sum() + (abs(model.j) > 1e-2).sum()
        nonzero_i.append(non_zero_params)

        bic[i, repeat] = (
            non_zero_params * (np.log(Yi.shape[0]) + 0.5 * np.log(model.n_params))
            - 2 * log_likelihood
        )
        # bic[i,repeat] = non_zero_params * (np.log(Yi.shape[0])) - 2*log_likelihood
    print(f"{i=}")
    print(f"λ={λ[i]}")
    print(f"Nonzero: {np.mean(nonzero_i)}")
    print(f"BIC: {np.mean(bic[i])}")

In [ ]:
fig, ax = plt.subplots(figsize=(3, 1.5), constrained_layout=True)
ax.plot(λ, bic.mean(axis=1), linewidth=0.75)
ax.scatter(λ, bic.mean(axis=1), s=3, linewidths=0.7, clip_on=False)

ax.set_xlim(1e-4, 10 ** (-1.8))
ax.set_xscale("log")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_xlabel(r"$\lambda$")
ax.set_ylabel("EBIC")
# ax.set_yscale("log")

In [ ]:
fig, ax = plt.subplots(figsize=(3, 2), constrained_layout=True)
ax.plot(λ, bic.mean(axis=1), linewidth=1)
ax.scatter(λ, bic.mean(axis=1), s=10, linewidths=0.7)
ax.set_xscale("log")
# ax.set_yscale("log")

In [ ]:
fig, ax = plt.subplots(figsize=(3, 2), constrained_layout=True)
ax.plot(λ[:7], bic.mean(axis=1)[:7], linewidth=1)
ax.scatter(λ[:7], bic.mean(axis=1)[:7], s=10, linewidths=0.7)
ax.set_xscale("log")
ax.set_yscale("log")

## Show how average sparsity changes

In [ ]:
rng = np.random.default_rng(RANDOM_SEED + 1)
repeats = 5
n_lambdas = 5
λ = np.logspace(-4, 0, n_lambdas, base=10)

X = np.ones((Y.shape[0], 2))

replicates = 1

k_mean = np.empty(n_lambdas, dtype=np.float64)
k_ci = np.empty(n_lambdas, dtype=np.float64)
for i in range(n_lambdas):
    print(i)
    k_i = np.empty(repeats, dtype=np.int64)
    for repeat in range(repeats):
        Ys = np.empty((Y.shape[0] * replicates, Y.shape[1], Y.shape[2]), dtype=np.int64)
        for j in range(replicates):
            _, Yj, _ = dataset.indices_to_numpy(
                "time-series", binarise=True, seed=RANDOM_SEED * repeat + j
            )
            Ys[Y.shape[0] * j : Y.shape[0] * (j + 1)] = Yj
        model = Ising.fit(
            Ys,
            update_method=UpdateMethod.SYNCHRONOUS,
            w=λ[i],
            rng=RANDOM_SEED + repeat,
            self_loops=True,
        )
        k_i[repeat] = (abs(model.param_vector()) > 1e-2).sum()

    k_mean[i] = k_i.mean()
    k_ci[i] = 1.96 * k_i.std(ddof=1)

In [ ]:
fig, ax = plt.subplots(figsize=(3, 2), constrained_layout=True)
# ax.plot(λ, k_mean, linewidth=1)
ax.errorbar(λ, y=k_mean, yerr=[k_ci, k_ci])
ax.scatter(λ, k_mean, s=10, linewidths=0.7)
ax.set_xscale("log")

In [ ]:
fig, ax = plt.subplots(figsize=(3, 2), constrained_layout=True)
# ax.plot(λ, k_mean, linewidth=1)
ax.errorbar(λ, y=k_mean, yerr=[k_ci, k_ci])
ax.scatter(λ, k_mean, s=10, linewidths=0.7)
ax.set_xscale("log")

In [ ]:
non_zero_params

In [ ]:
fig, axes = plt.subplots(ncols=3, figsize=(10, 2.5))

layout_model = Ising.fit(
    Y, update_method=UpdateMethod.SYNCHRONOUS, w=0, rng=RANDOM_SEED, self_loops=True
)
for i, λ in enumerate((0.01, 0.025, 0.05)):
    model = Ising.fit(
        Y, update_method=UpdateMethod.SYNCHRONOUS, w=λ, rng=RANDOM_SEED, self_loops=True
    )
    model.adj[abs(model.j) < 0.05] = False
    model.draw(
        use_layout_from=layout_model, ax=axes[i], vlim_h=(-0.5, 0.5), vlim_j=(-1, 1)
    )